# KG1 v141 — Colab Pro+ A100/H100 (TODOS DATASETS + HYPERPARAMS AJUSTADOS)

## Objetivo: TARGET 0.92 (consensus 28 APIs)

### Dataset combinado (~11,400 CoTs):
| Source | Samples | Tipo |
|---|---|---|
| Tong corpus (`felipesp1983/nemotron-cot-tong-dgxchen`) | 6,014 | bit/cipher/gravity/numeral/unit + transformation |
| Cryptarithm 813 (`felipesp1983/nemotron-cryptarithm-cot-813`) | 813 | symbolic deduce + guess |
| Equation_numeric_guess 150 (gist DeepSeek-R1 distill) | 150 | numeric guess |
| **Z3 synth verified** (`felipesp1983/nemotron-cryptarithm-synth-4425`) | **4,425** | **2,198 deduce + 1,227 guess + 1,000 eq_numeric** |
| **TOTAL** | **~11,402** | **gap-aware focus** |

### Hyperparams (consenso triple check 28 APIs):
- LoRA r=32 alpha=32 all-linear+lm_head (Tong validated)
- **lr=5e-5** (era 2e-4, consenso 23 APIs lower for synth+distill)
- **epochs=2** (era 1, consenso 18 APIs - more for synth convergence)
- max_length=8192 (Kaggle limit, NÃO mudar)
- batch effective 32 (grad_accum=32, micro=1)
- Chat template enable_thinking=True
- Prompt masking labels=-100

### Score expectations (consenso):
- Mediana 28 APIs: **0.92**
- Range: [0.88 - 0.95]
- P(>= 0.87): 75-85%

## Setup Colab Secrets
Adicione 3 secrets no Colab (icone chave lateral esquerdo):
- `HF_TOKEN` (HuggingFace, com permissoes Write+Jobs - so necessario no Cell 9)
- `KAGGLE_USERNAME` (Felipe enviou separadamente)
- `KAGGLE_KEY` (Felipe enviou separadamente)

Runtime: **A100 HighRAM (40GB) NF4** OU **H100/A100 80GB BF16**
- Em A100 40GB → notebook auto-aplica NF4 quantization (model 30B em 4-bit cabe em ~17GB)
- Em A100/H100 80GB → BF16 nativo (~63GB)
- Runtime → Change runtime type → A100 HighRAM ou H100

In [ ]:
# CELL 1: Setup secrets + verify GPU
import os
from google.colab import userdata

# HF token: usa HF_KEY (preferred) com fallback para HF_TOKEN
# Token e necessario SOMENTE no final (Cell 9) para upload do adapter
hf_token = None
for key_name in ['HF_KEY', 'HF_TOKEN']:
    try:
        hf_token = userdata.get(key_name)
        if hf_token:
            os.environ['HF_TOKEN'] = hf_token
            print(f'HF token configured via {key_name} (length: {len(hf_token)})')
            break
    except Exception:
        continue
if not hf_token:
    print('[WARN] HF_KEY/HF_TOKEN nao configurado nos secrets')
    print('Configure antes do Cell 9 (upload adapter)')

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print(f'Kaggle creds: {os.environ["KAGGLE_USERNAME"]}')
except Exception as e:
    print(f'[INFO] Kaggle not set (optional for training): {e}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

import psutil
print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB')

In [ ]:
# CELL 2: Install deps (Colab has CUDA + PyTorch preinstalled)
%pip install -q --upgrade transformers>=4.48.0 peft>=0.14.0 trl>=0.14.0 datasets>=3.0.0 accelerate>=1.3.0 bitsandbytes>=0.45.0 huggingface_hub>=0.27.0
%pip install -q einops packaging ninja triton

import transformers, peft, trl
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'trl: {trl.__version__}')

In [ ]:
# CELL 3: Install mamba-ssm + causal-conv1d (Colab usually OK)
%pip install -q mamba-ssm causal-conv1d

# Verify
import importlib, sys
try:
    import mamba_ssm
    print(f'mamba_ssm: {mamba_ssm.__version__}')
    import causal_conv1d
    print('causal_conv1d: imported')
    import selective_scan_cuda
    print('[OK] selective_scan_cuda kernel')
    import causal_conv1d_cuda
    print('[OK] causal_conv1d_cuda kernel')
except ImportError as e:
    print(f'[WARN] {e}')
    print('Trying NVIDIA wheel index...')
    !pip install -q --extra-index-url https://pypi.nvidia.com mamba-ssm causal-conv1d
    import mamba_ssm
    print(f'mamba_ssm via nvidia: {mamba_ssm.__version__}')

In [ ]:
# CELL 4: Download ALL datasets via direct HTTP (sem precisar de token HF)
import urllib.request
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}

for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    print(f'Downloading {name}...')
    urllib.request.urlretrieve(url, out)
    sz = os.path.getsize(out)
    print(f'  [OK] {name}: {sz/1e6:.2f} MB' if sz > 1e6 else f'  [OK] {name}: {sz/1e3:.1f} KB')

tong_path = os.path.join(DATA_DIR, 'tong.csv')
crypt_path = os.path.join(DATA_DIR, 'cryptarithm_813.jsonl')
eq_guess_path = os.path.join(DATA_DIR, 'eq_guess_150.jsonl')
synth_path = os.path.join(DATA_DIR, 'synth_4425.jsonl')

In [ ]:
# CELL 5: Format + Merge ALL datasets
import csv, json
from datasets import Dataset

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

all_data = []

# Tong (6014)
n_tong = 0
with open(tong_path, encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'tong'})
            n_tong += 1

# Cryptarithm 813
n_crypt = 0
with open(crypt_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid') and row.get('cot'):
            all_data.append({'prompt': row['prompt'].strip() + PROMPT_SUFFIX,
                             'completion': row['cot'].strip(), 'source': 'cryptarithm_813'})
            n_crypt += 1

# eq_guess 150
n_eq = 0
with open(eq_guess_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'eq_guess_150'})
            n_eq += 1

# Synth 4425 (NEW)
n_synth = 0
with open(synth_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'synth_4425'})
            n_synth += 1

print(f'Tong:               {n_tong}')
print(f'Cryptarithm 813:    {n_crypt}')
print(f'EqGuess 150:        {n_eq}')
print(f'Synth Z3 4425:      {n_synth}')
print(f'TOTAL:              {len(all_data)}')

# CURRICULUM: easy first (Tong) -> hard (synth + cryptarithm)
# Sort by source priority for curriculum learning
SOURCE_ORDER = {'tong': 0, 'eq_guess_150': 1, 'cryptarithm_813': 2, 'synth_4425': 3}
all_data.sort(key=lambda x: SOURCE_ORDER.get(x['source'], 99))

ds = Dataset.from_list(all_data)
print(f'\nCurriculum order: tong -> eq_guess -> cryptarithm_813 -> synth_4425')

In [ ]:
# CELL 6: Load Nemotron-3-Nano-30B + LoRA (auto NF4 if VRAM < 70GB)
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

# Auto-detect VRAM and decide quantization
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_NF4 = vram_gb < 70  # H100 80GB / A100 80GB = no NF4; A100 40GB = NF4 needed
print(f'VRAM: {vram_gb:.1f} GB -> USE_NF4: {USE_NF4}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.time()
if USE_NF4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map={'': 0}, trust_remote_code=True,
        token=os.environ['HF_TOKEN'], attn_implementation='eager',
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=os.environ['HF_TOKEN'],
        attn_implementation='eager',
    )
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()

print(f'[OK] Model in {time.time()-t0:.1f}s, VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=32, target_modules=TARGET_REGEX,
    lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'[OK] LoRA trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B')
print(f'VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB')

In [ ]:
# CELL 7: Tokenize com chat template + prompt masking
MAX_LENGTH = 8192

def fmt(ex):
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']}
    ]
    prompt_ids = tokenizer.apply_chat_template(
        [messages[0]], tokenize=True, add_generation_prompt=True, enable_thinking=True)
    full_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, enable_thinking=True)
    if len(full_ids) > MAX_LENGTH:
        full_ids = full_ids[:MAX_LENGTH]
    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'[OK] Tokenized: {len(tokenized)} samples')

In [ ]:
# CELL 8: Train (lr=5e-5, epochs=2 - consenso triple check)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = '/content/output_v141'
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,             # CONSENSO 18 APIs (era 1 no Tong)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32, # batch effective 32
    learning_rate=5e-5,             # CONSENSO 23 APIs (era 2e-4 no Tong)
    lr_scheduler_type='cosine',     # cosine (em vez de linear) para 2 epochs
    warmup_ratio=0.03,              # warmup leve para 2 epochs
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.01,              # leve regularização para synth data
    max_grad_norm=1.0,
    logging_steps=10, save_steps=200, save_total_limit=3,
    bf16=True, optim='adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    remove_unused_columns=False, report_to='none', dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding=True, label_pad_token_id=-100,
    pad_to_multiple_of=8, return_tensors='pt',
)
trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=data_collator)

import time
t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print(f'[OK] Training in {train_time_min:.1f} min')

In [ ]:
# CELL 9: Save adapter + Upload to HF
from huggingface_hub import HfApi, create_repo

ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# List files saved
for f in os.listdir(ADAPTER_DIR):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f}: {sz/1e6:.2f} MB')

DEST_REPO = 'felipesp1983/kg1-v141-final'
api = HfApi(token=os.environ['HF_TOKEN'])
create_repo(DEST_REPO, private=False, exist_ok=True, token=os.environ['HF_TOKEN'])
api.upload_folder(
    folder_path=ADAPTER_DIR, repo_id=DEST_REPO, repo_type='model',
    commit_message=f'v141 trained {train_time_min:.1f}min Colab A100 (Tong+813+150+4425synth lr5e-5 ep2)',
)
print(f'[OK] Uploaded: https://huggingface.co/{DEST_REPO}')

In [ ]:
# CELL 10: GDrive backup
from google.colab import drive
drive.mount('/content/drive')

import shutil
GDRIVE_BACKUP = '/content/drive/My Drive/KG1_v141_adapter'
os.makedirs(GDRIVE_BACKUP, exist_ok=True)
shutil.copytree(ADAPTER_DIR, GDRIVE_BACKUP, dirs_exist_ok=True)
print(f'[OK] Backup: {GDRIVE_BACKUP}')

import subprocess
sz = subprocess.run(['du', '-sh', GDRIVE_BACKUP], capture_output=True, text=True).stdout
print(f'Size: {sz}')

## Próximos passos após v141 treinar

### 1. Quick eval local (no Colab)
```python
# Test adapter on a few prompts
from transformers import pipeline
# ... eval pequeno script
```

### 2. Submit to Kaggle (no terminal local)
```bash
python scripts/submit_kaggle.py \
    --hf-repo felipesp1983/kg1-v141-final \
    --reference-adapter-dir runs/attached_085_audit_20260416 \
    --test-csv data/kaggle/unzipped/test.csv \
    --message "v141 11400CoTs Tong+synth4425+eq150 lr5e-5 ep2" \
    --max-daily-submits 5
```

### 3. Score esperado (consenso 28 APIs)
- Mediana: **0.92**
- Range: [0.88 - 0.95]
- P(>= 0.87): 75-85%

### 4. Se v141 < 0.87
Próximo passo: **v142 GRPO** sobre v141 SFT
- Adaptar `rewards.py` do andy279 (open-r1 fork)
- Verifiable reward via Z3 (cryptarithm) + sympy (eq_numeric)
- +10% adicional esperado